# Pandas 专项训练：缺失值识别与标准化

## 练习主题

识别并统一业务数据中的真实缺失值与伪装缺失值。

## 业务背景

某设备运维系统导出了一批巡检记录。

由于数据来自人工录入和不同系统，部分字段存在：

- 前后空格
- 大小写不统一
- 空字符串
- `N/A`
- `NULL`
- `-`
- `unknown`
- Python 原生缺失值

这些值必须先统一处理，才能正确统计缺失情况。

## 训练目标

1. 保留原始数据，创建清洗副本。
2. 检查清洗前 Pandas 能直接识别的缺失值。
3. 清除文本字段前后的空格。
4. 统一设备编号、站点编号和状态字段的大小写。
5. 将伪装缺失值统一转换为 `pd.NA`。
6. 生成字段级缺失情况报告。
7. 根据字段业务含义分别使用删除、填充或保留策略。
8. 验证清洗结果是否符合要求。

## 清洗规则

### 文本格式标准化

- `device_id`：清除空格并转换为大写。
- `site`：清除空格并转换为大写。
- `status`：清除空格并转换为大写。
- `signal_strength`：只清除前后空格，暂时不转换数值类型。
- `technician`：只清除前后空格。

### 缺失值标准化

下列内容均视为缺失值：

- 空字符串
- 只有空格的字符串
- `N/A`
- `NULL`
- `-`
- `UNKNOWN`

将它们统一转换为 `pd.NA`。

### 业务处理规则

- 删除 `device_id` 缺失的记录。
- `status` 缺失时填充为 `UNKNOWN`。
- `technician` 缺失时填充为 `UNASSIGNED`。
- `signal_strength` 缺失值暂时保留，不删除、不填充。

## 限制条件

- 不允许手工逐行修改数据。
- 不允许直接修改原始 DataFrame。
- 不允许对整个 DataFrame 直接执行 `dropna()`。
- 必须使用 Pandas 向量化方法完成清洗。

In [72]:
import pandas as pd

data = {
    "record_id": range(1, 13),

    "device_id": [
        "r34-01",
        "r34-01",
        "r34-02",
        "r34-02",
        " r34-03 ",
        "r34-03",
        None,
        "r34-04",
        "r34-04",
        "r34-05",
        "r34-05",
        "r34-06"
    ],

    "site": [
        "R34",
        " R34 ",
        "r34",
        "R34",
        "R35",
        "R35",
        "R36",
        " R36 ",
        "R36",
        "R37",
        "r37",
        "R38"
    ],

    "status": [
        "NORMAL",
        " normal ",
        "ERROR",
        " ",
        "N/A",
        None,
        "NORMAL",
        "NULL",
        "ERROR",
        "-",
        "normal",
        " ERROR "
    ],

    "signal_strength": [
        "18.5",
        "19.1",
        " ",
        "17.8",
        "unknown",
        None,
        "20.0",
        "21.2",
        "NULL",
        "19.9",
        "-",
        "18.7"
    ],

    "technician": [
        "Li",
        " Li ",
        "Wang",
        "Wang",
        "",
        "N/A",
        "Chen",
        "Chen",
        "NULL",
        "Zhao",
        "zhao",
        "-"
    ],

    "inspect_time": [
        "2026-07-01 08:00",
        "2026-07-01 08:10",
        "2026-07-01 08:20",
        "2026-07-01 08:30",
        "2026-07-01 08:40",
        "2026-07-01 08:50",
        "2026-07-01 09:00",
        "2026-07-01 09:10",
        "2026-07-01 09:20",
        "2026-07-01 09:30",
        "2026-07-01 09:40",
        "2026-07-01 09:50"
    ]
}

df_raw = pd.DataFrame(data)

df_raw

,record_id,device_id,site,status,signal_strength,technician,inspect_time
0,1,r34-01,R34,NORMAL,18.5,Li,2026-07-01 08:00
1,2,r34-01,R34,normal,19.1,Li,2026-07-01 08:10
2,3,r34-02,r34,ERROR,,Wang,2026-07-01 08:20
3,4,r34-02,R34,,17.8,Wang,2026-07-01 08:30
4,5,r34-03,R35,N/A,unknown,,2026-07-01 08:40
5,6,r34-03,R35,NaN,NaN,N/A,2026-07-01 08:50
6,7,NaN,R36,NORMAL,20.0,Chen,2026-07-01 09:00
7,8,r34-04,R36,NULL,21.2,Chen,2026-07-01 09:10
8,9,r34-04,R36,ERROR,NULL,NULL,2026-07-01 09:20
9,10,r34-05,R37,-,19.9,Zhao,2026-07-01 09:30


### 一、检查数据原始表达

In [73]:
# 创建原始字段检查表

check_cols = [
    'device_id',
    'site',
    'status',
    'signal_strength',
    'technician',
    'inspect_time'
]

In [74]:
# 1. # 检查数据规模，为清洗前后的行列数量对比建立基准
df_raw.shape

(12, 7)

In [75]:
# 2. 检查字段类型、非空数量和数据整体结构
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   record_id        12 non-null     int64
 1   device_id        11 non-null     str  
 2   site             12 non-null     str  
 3   status           11 non-null     str  
 4   signal_strength  11 non-null     str  
 5   technician       12 non-null     str  
 6   inspect_time     12 non-null     str  
dtypes: int64(1), str(6)
memory usage: 1.2 KB


In [76]:
# 3. 统计 Pandas 当前已经识别的真实缺失值
df_raw.isna().sum()

record_id          0
device_id          1
site               0
status             1
signal_strength    1
technician         0
inspect_time       0
dtype: int64

In [77]:
# 4.检查重点字段的原始表达方式及其出现次数
for col in check_cols:
    print(f"\n===== {col} =====")
    print(df_raw[col].map(repr).value_counts())



===== device_id =====
device_id
'r34-01'      2
'r34-02'      2
'r34-04'      2
'r34-05'      2
' r34-03 '    1
'r34-03'      1
nan           1
'r34-06'      1
Name: count, dtype: int64

===== site =====
site
'R34'      2
'R35'      2
'R36'      2
' R34 '    1
'r34'      1
' R36 '    1
'R37'      1
'r37'      1
'R38'      1
Name: count, dtype: int64

===== status =====
status
'NORMAL'      2
'ERROR'       2
' normal '    1
' '           1
'N/A'         1
nan           1
'NULL'        1
'-'           1
'normal'      1
' ERROR '     1
Name: count, dtype: int64

===== signal_strength =====
signal_strength
'18.5'       1
'19.1'       1
' '          1
'17.8'       1
'unknown'    1
nan          1
'20.0'       1
'21.2'       1
'NULL'       1
'19.9'       1
'-'          1
'18.7'       1
Name: count, dtype: int64

===== technician =====
technician
'Wang'    2
'Chen'    2
'Li'      1
' Li '    1
''        1
'N/A'     1
'NULL'    1
'Zhao'    1
'zhao'    1
'-'       1
Name: count, dtype: int64

=

### 二、创建清洗副本

In [78]:
# 创建清洗副本
df_cleaning = df_raw.copy()


### 三、将伪缺失值统一为pd.NA

In [79]:
# 将伪缺失值统一为 pd.NA
missing_markers = {
    "": pd.NA,
    "N/A": pd.NA,
    "NULL": pd.NA,
    "-": pd.NA,
    "UNKNOWN": pd.NA
}

### 四、清洗 `device_id`

In [80]:
# 标准化 device_id
df_cleaning["device_id"] = (
    df_cleaning["device_id"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace(missing_markers)
)
df_cleaning['device_id']

0     R34-01
1     R34-01
2     R34-02
3     R34-02
4     R34-03
5     R34-03
6       <NA>
7     R34-04
8     R34-04
9     R34-05
10    R34-05
11    R34-06
Name: device_id, dtype: string

In [81]:
# 保存因 device_id 缺失而准备删除的记录
removed_missing_device = df_cleaning.loc[
    df_cleaning['device_id'].isna()
].copy()
removed_missing_device

,record_id,device_id,site,status,signal_strength,technician,inspect_time
6,7,<NA>,R36,NORMAL,20.0,Chen,2026-07-01 09:00


In [82]:
# 删除 device_id 缺失的整行
df_cleaning = (
    df_cleaning
    .dropna(subset=["device_id"])
    .reset_index(drop=True)
)
df_cleaning

,record_id,device_id,site,status,signal_strength,technician,inspect_time
0,1,R34-01,R34,NORMAL,18.5,Li,2026-07-01 08:00
1,2,R34-01,R34,normal,19.1,Li,2026-07-01 08:10
2,3,R34-02,r34,ERROR,,Wang,2026-07-01 08:20
3,4,R34-02,R34,,17.8,Wang,2026-07-01 08:30
4,5,R34-03,R35,N/A,unknown,,2026-07-01 08:40
5,6,R34-03,R35,NaN,NaN,N/A,2026-07-01 08:50
6,8,R34-04,R36,NULL,21.2,Chen,2026-07-01 09:10
7,9,R34-04,R36,ERROR,NULL,NULL,2026-07-01 09:20
8,10,R34-05,R37,-,19.9,Zhao,2026-07-01 09:30
9,11,R34-05,r37,normal,-,zhao,2026-07-01 09:40


### 五、清洗 `site`

In [83]:
# 1.转为 `string` 类型、去除首尾空格、统一大小写

df_cleaning['site'] = (
    df_cleaning['site']
    .astype('string')
    .str.strip()
    .str.upper()
)
df_cleaning['site']

0     R34
1     R34
2     R34
3     R34
4     R35
5     R35
6     R36
7     R36
8     R37
9     R37
10    R38
Name: site, dtype: string

In [84]:
# 检查是否存在缺失值和异常站点

# 1. 查看清洗后的站点分布
print(f"查看清洗后的站点分布:\n{df_cleaning["site"].value_counts(dropna=False)}")

# 2. 检查缺失值
missing_site_mask = df_cleaning["site"].isna()

print("\nsite 缺失数量：", missing_site_mask.sum())

print("\nsite 缺失记录：")
display(
    df_cleaning.loc[
        missing_site_mask
    ]
)

# 3. 检查编号格式：R + 两位数字
invalid_site_format_mask = (
    df_cleaning["site"].notna()
    & ~df_cleaning["site"].str.fullmatch(r"R\d{2}")
)

print("\n格式异常数量：", invalid_site_format_mask.sum())

print("\n格式异常记录：")
display(
    df_cleaning.loc[
        invalid_site_format_mask
    ]
)

# 4. 检查业务合法范围
valid_sites = {"R34", "R35", "R36", "R37", "R38"}

invalid_site_value_mask = (
    df_cleaning["site"].notna()
    & ~df_cleaning["site"].isin(valid_sites)
)

print("\n不在合法站点清单中的数量：", invalid_site_value_mask.sum())

print("\n不在合法站点清单中的记录：")
display(
    df_cleaning.loc[
        invalid_site_value_mask
    ]
)

# 5.验证
assert df_cleaning["site"].notna().all(), "site 字段仍存在缺失值"
assert not invalid_site_format_mask.any(), "存在格式异常的站点编号"
assert not invalid_site_value_mask.any(), "存在非法站点编号"

查看清洗后的站点分布:
site
R34    4
R35    2
R36    2
R37    2
R38    1
Name: count, dtype: int64[pyarrow]

site 缺失数量： 0

site 缺失记录：


,record_id,device_id,site,status,signal_strength,technician,inspect_time



格式异常数量： 0

格式异常记录：


,record_id,device_id,site,status,signal_strength,technician,inspect_time



不在合法站点清单中的数量： 0

不在合法站点清单中的记录：


,record_id,device_id,site,status,signal_strength,technician,inspect_time


### 六、清洗 `status`

In [ ]:
# 1.转为 `string` 类型、去除首尾空格、统一大小写、规范化空值

df_cleaning['status'] = (
    df_cleaning['status']
    .astype('string')
    .str.strip()
    .str.upper()
    .replace(missing_markers)
)
df_cleaning['status']

0     NORMAL
1     NORMAL
2      ERROR
3       <NA>
4       <NA>
5       <NA>
6       <NA>
7      ERROR
8       <NA>
9     NORMAL
10     ERROR
Name: status, dtype: string

In [86]:
# 2.检查标准化后识别出的真实缺失值数量
df_cleaning["status"].isna().sum()

np.int64(5)

In [89]:
# 3.检查标准化后的状态类别及其数量
df_cleaning["status"].value_counts(dropna=False)



status
<NA>      5
NORMAL    3
ERROR     3
Name: count, dtype: int64[pyarrow]

In [90]:
# 4.验证：

assert(
    df_cleaning['status']
    .dropna()
    .isin({'NORMAL','ERROR'})
    .all()
    
),"status 中存在未预期的状态值"

### 七.清洗 `singal_strength`

In [94]:
# 1.转为 `string` 类型、去除首尾空格、统一大小写、规范化空值
df_cleaning['signal_strength'] = (
    df_cleaning['signal_strength'] 
    .astype('string')
    .str.strip()
    .str.upper()
    .replace(missing_markers)
    .astype('float')
)

In [93]:
# 2.检查标准化后识别出的真实缺失值数量
df_cleaning['signal_strength'].isna().sum()

np.int64(5)

In [95]:
# 3.检查转换后的数据类型
df_cleaning["signal_strength"].dtype

dtype('float64')

In [96]:
# 4.检查非缺失数量、均值、标准差、分位数和极值
df_cleaning["signal_strength"].describe()

count     6.00
mean     19.20
std       1.20
min      17.80
25%      18.55
50%      18.90
75%      19.70
max      21.20
Name: signal_strength, dtype: float64

### 八.清洗 `technician`

In [99]:
# 1.转为 `string` 类型、去除首尾空格、统一大小写、规范化空值
df_cleaning['technician'] = (
    df_cleaning['technician'] 
    .astype('string')
    .str.strip()
    .str.upper()
    .replace(missing_markers)
)


In [100]:
# # 2.统计标准化后识别出的真实缺失值数量

df_cleaning['technician'].isna().sum()

np.int64(4)

In [101]:
# 查看标准化后的状态类别及其数量
df_cleaning['technician'].value_counts(dropna=False)

technician
<NA>    4
LI      2
WANG    2
ZHAO    2
CHEN    1
Name: count, dtype: int64[pyarrow]

### 九.清洗 `inspect_time`

In [102]:
# 1.转换为日期时间类型；无法解析的值统一转为 NaT
df_cleaning["inspect_time"] = pd.to_datetime(
    df_cleaning["inspect_time"],
    format="%Y-%m-%d %H:%M",
    errors="coerce"
)



In [103]:
# 2.检查转换后的类型
df_cleaning["inspect_time"].dtype

dtype('<M8[us]')

In [104]:
# 检查是否存在转换失败或原本缺失的时间
df_cleaning["inspect_time"].isna().sum()

np.int64(0)

In [107]:
df_cleaning

,record_id,device_id,site,status,signal_strength,technician,inspect_time
0,1,R34-01,R34,NORMAL,18.5,LI,2026-07-01 08:00:00
1,2,R34-01,R34,NORMAL,19.1,LI,2026-07-01 08:10:00
2,3,R34-02,R34,ERROR,NaN,WANG,2026-07-01 08:20:00
3,4,R34-02,R34,<NA>,17.8,WANG,2026-07-01 08:30:00
4,5,R34-03,R35,<NA>,NaN,<NA>,2026-07-01 08:40:00
5,6,R34-03,R35,<NA>,NaN,<NA>,2026-07-01 08:50:00
6,8,R34-04,R36,<NA>,21.2,CHEN,2026-07-01 09:10:00
7,9,R34-04,R36,ERROR,NaN,<NA>,2026-07-01 09:20:00
8,10,R34-05,R37,<NA>,19.9,ZHAO,2026-07-01 09:30:00
9,11,R34-05,R37,NORMAL,NaN,ZHAO,2026-07-01 09:40:00


### 十.缺失统计报告

In [114]:
missing_report = (
    df_cleaning
    .isna()
    .agg(['sum','mean'])
    .T
    .rename(
        columns={
            'sum':'missing_count',
            'mean':'missing_rate'
        }
    )
    .reset_index(names="column")
    .sort_values(
        by=["missing_count", "column"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)
# 调整统计结果的显示格式
missing_report["missing_count"] = (
    missing_report["missing_count"]
    .astype("int64")
)

missing_report["missing_rate"] = (
    missing_report["missing_rate"]
    .round(4)
)

missing_report

,column,missing_count,missing_rate
0,signal_strength,5,0.4545
1,status,5,0.4545
2,technician,4,0.3636
3,device_id,0,0.0000
4,inspect_time,0,0.0000
5,record_id,0,0.0000
6,site,0,0.0000


### 十一.根据业务含义处理缺失值

In [115]:
# 根据字段业务含义处理缺失值
df_cleaning["status"] = (
    df_cleaning["status"]
    .fillna("UNKNOWN")
)

df_cleaning["technician"] = (
    df_cleaning["technician"]
    .fillna("UNASSIGNED")
)
df_cleaning

,record_id,device_id,site,status,signal_strength,technician,inspect_time
0,1,R34-01,R34,NORMAL,18.5,LI,2026-07-01 08:00:00
1,2,R34-01,R34,NORMAL,19.1,LI,2026-07-01 08:10:00
2,3,R34-02,R34,ERROR,NaN,WANG,2026-07-01 08:20:00
3,4,R34-02,R34,UNKNOWN,17.8,WANG,2026-07-01 08:30:00
4,5,R34-03,R35,UNKNOWN,NaN,UNASSIGNED,2026-07-01 08:40:00
5,6,R34-03,R35,UNKNOWN,NaN,UNASSIGNED,2026-07-01 08:50:00
6,8,R34-04,R36,UNKNOWN,21.2,CHEN,2026-07-01 09:10:00
7,9,R34-04,R36,ERROR,NaN,UNASSIGNED,2026-07-01 09:20:00
8,10,R34-05,R37,UNKNOWN,19.9,ZHAO,2026-07-01 09:30:00
9,11,R34-05,R37,NORMAL,NaN,ZHAO,2026-07-01 09:40:00


### 十二、整体验证

In [116]:
# 验证被删除记录的数量
assert len(removed_missing_device) == 1, \
    "因 device_id 缺失而删除的记录数量异常"

# 验证被删除记录的 device_id 确实为空
assert removed_missing_device["device_id"].isna().all(), \
    "被删除记录中存在非缺失的 device_id"

# 验证清洗表与删除记录表的行数之和等于原始数据
assert (
    len(df_cleaning) + len(removed_missing_device)
    == len(df_raw)
), "清洗后记录与删除记录无法完整对应原始数据"

# 验证所有 record_id 都可以追溯到原始数据
processed_record_ids = set(df_cleaning["record_id"])
removed_record_ids = set(removed_missing_device["record_id"])
raw_record_ids = set(df_raw["record_id"])

assert (
    processed_record_ids | removed_record_ids
    == raw_record_ids
), "清洗过程中存在记录丢失或新增"

print("删除记录和原始数据的对应关系验证通过。")

删除记录和原始数据的对应关系验证通过。
